# S3 Parte 2 (opcional) - Preparar `transactions.parquet` para series de tiempo

**Actividad:** aplicar los mismos controles de calidad de S3 (esquema, duplicados) a `transactions.parquet` — el dataset de transacciones reales H&M, con fecha (`t_dat`) — y escribirlo particionado por mes, dejándolo listo como entrada directa de S4 (ML distribuido / series de tiempo).

**Por que es opcional:** `transactions.parquet` tiene 31 788 324 filas — unas 23 veces mas que `customers.csv`. Completa esto solo si ya terminaste 3.1-3.12 de la guia principal y tu equipo cuenta con los recursos de computo (memoria y tiempo) para procesar un dataset de este tamano.

**Producto de esta parte:** `transactions_particionado/` en Parquet, particionado por mes (`anio_mes`), verificado por lectura y por *partition pruning*.

Guia completa: `docs/sesiones/S03_Parte2_Transacciones_Particionadas.md`.

## Preparar los datos

**Producto del paso:** `transactions.parquet` disponible en `pyspark/sesiones/s03-parte2-transacciones/data/`.

Ya lo descargaste en S2 — copialo a la carpeta de esta parte (desde tu maquina, no dentro del notebook):

```bash
cp lambda26/pyspark/sesiones/s02-fundamentos/data/transactions.parquet lambda26/pyspark/sesiones/s03-parte2-transacciones/data/
```

Pesa ~773 MB — la copia tarda mas que la de `customers.csv` en S3 (~207 MB). **Espera a que termine** antes de correr el notebook; confirma con el tamano del archivo:

```bash
ls -la lambda26/pyspark/sesiones/s02-fundamentos/data/transactions.parquet
ls -la lambda26/pyspark/sesiones/s03-parte2-transacciones/data/transactions.parquet
```

Ambos deben coincidir exactamente en tamano antes de seguir.

## Crear la `SparkSession`

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion3-parte2-transacciones")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "6g")
    .getOrCreate()
)

spark

`spark.sql.shuffle.partitions` sube de 8 (S3) a 16 — el dataset es mas grande, mas particiones evitan que cada una cargue demasiados datos en el `groupBy()`/`distinct()` que vienen mas abajo. `spark.driver.memory` tambien sube, de 4g (S3) a **6g**: en una corrida real, el `distinct()` y el `groupBy()` de mas abajo (ambos sobre las 31.7M filas completas) llegaron a lanzar `OutOfMemoryError: Java heap space` con 4g — Spark reintento solo y termino completando, pero no hay que confiar en que siempre se recupere. Si tu maquina tiene poca memoria disponible (Docker compitiendo con otros contenedores, S3 principal, etc.), cierra lo que no necesites antes de correr esta parte.

In [ ]:
ORIGEN_DATOS = "/opt/s03-parte2-transacciones/data"
ARTIFACTS = "/opt/s03-parte2-transacciones/artifacts"

## Cargar `transactions.parquet` y validar

**Producto del paso:** `df_transactions` cargado, con columnas y conteo confirmados.

A diferencia de `customers.csv` (S3, un CSV sin tipos propios), Parquet ya guarda su propio esquema — no hace falta `StructType` explicito para *leer* correctamente. Pero eso no reemplaza la validacion de columnas (mismo control de calidad #1 de S3, 2.2):

In [ ]:
df_transactions = spark.read.parquet(f"{ORIGEN_DATOS}/transactions.parquet")

df_transactions.printSchema()
df_transactions.count()

In [ ]:
df_transactions.show(5, truncate=False)

In [ ]:
columnas_requeridas = {"t_dat", "customer_id", "article_id", "price", "sales_channel_id"}

faltantes = columnas_requeridas - set(df_transactions.columns)
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(faltantes)}")

print("Esquema validado: las 5 columnas requeridas estan presentes.")

## Nulos y duplicados

**Producto del paso:** confirmacion de que no hay nulos, y diagnostico de duplicados con dos definiciones distintas.

Mismo control que en S3 (2.6), sobre un dataset distinto:

In [ ]:
from pyspark.sql.functions import col, count, when

df_transactions.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_transactions.columns
]).show()

En una corrida real, dio 0 en las 5 columnas — a diferencia de `customers.csv` en S3 (FN/Active con ~65% de nulos), `transactions.parquet` no tiene nulos. No todo dataset necesita `.na.fill()`/`.na.drop()`; el control se hace igual, el resultado puede ser "no hay nada que tratar".

Duplicados exactos (todas las columnas) — si una misma linea de venta quedo registrada dos veces, probablemente sea un problema de ingesta:

In [ ]:
total = df_transactions.count()
sin_duplicados = df_transactions.distinct().count()

print(f"Total: {total}, sin duplicar (fila completa): {sin_duplicados}, duplicados exactos: {total - sin_duplicados}")

Un segundo tipo de "duplicado", distinto: el mismo cliente comprando el mismo articulo el mismo dia, mas de una vez. Esto **no** es necesariamente un error — un cliente puede llevarse dos unidades en compras separadas el mismo dia — asi que se diagnostica (2.5, `groupBy()+count()`), no se elimina a ciegas:

In [ ]:
mismo_cliente_articulo_dia = (
    df_transactions
    .groupBy("t_dat", "customer_id", "article_id")
    .count()
    .filter(col("count") > 1)
)

print(f"Combinaciones (fecha, cliente, articulo) con mas de una fila: {mismo_cliente_articulo_dia.count()}")
mismo_cliente_articulo_dia.orderBy(col("count").desc()).show(5)

## Validar la fecha y preparar la columna de particion

**Producto del paso:** `t_dat` convertido a tipo fecha, y una columna `anio_mes` derivada para particionar.

`t_dat` llega como `string` (`"2018-09-20"`), no como fecha — aunque el archivo sea Parquet, el tipo real de esta columna sigue siendo texto. Para cualquier operacion de fecha real (la que va a hacer falta en S4) hay que convertirlo explicitamente:

In [ ]:
from pyspark.sql.functions import to_date, date_format

df_transactions = df_transactions.withColumn("t_dat", to_date(col("t_dat"), "yyyy-MM-dd"))

df_transactions.printSchema()
df_transactions.select("t_dat").summary("min", "max").show()

25 meses distintos (verificado) es la cardinalidad correcta para particionar (2.7, S3) — particionar por dia (734 valores distintos) seria demasiadas carpetas para el beneficio que aporta; particionar por `customer_id` seria el mismo error de alta cardinalidad que ya viste en S3. `anio_mes` (formato `"2018-09"`) es la columna de particion:

In [ ]:
df_transactions = df_transactions.withColumn("anio_mes", date_format(col("t_dat"), "yyyy-MM"))

df_transactions.select("t_dat", "anio_mes").show(5)
df_transactions.select("anio_mes").distinct().orderBy("anio_mes").count()

## Escritura particionada por mes

**Producto del paso:** `transactions_particionado/` en Parquet, con una carpeta por mes.

In [ ]:
(
    df_transactions
    .repartition("anio_mes")
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("anio_mes")
    .save(f"{ARTIFACTS}/transactions_particionado")
)

In [ ]:
import os

carpetas = sorted(os.listdir(f"{ARTIFACTS}/transactions_particionado"))
print(f"{len(carpetas)} carpetas:")
for c in carpetas:
    print(c)

## Leer de vuelta y verificar *partition pruning*

**Producto del paso:** confirmacion de que la salida particionada se lee bien y que filtrar por mes evita escanear el dataset completo — la razon de fondo por la que se particiono.

In [ ]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/transactions_particionado")
df_verificacion.printSchema()
df_verificacion.show(5, truncate=False)

assert df_verificacion.count() == df_transactions.count()
print("Conteo verificado: coincide con el original.")

Filtra por un solo mes y revisa el plan — deberias ver `PartitionFilters`, igual que en S3 (2.9, 3.11), pero acá con una diferencia práctica real: sin particionamiento, esta consulta tendría que escanear los 31.7 millones de filas; con `anio_mes`, Spark ignora directamente las otras 24 carpetas:

In [ ]:
df_verificacion.filter(col("anio_mes") == "2019-11").explain(True)

In [ ]:
ventas_noviembre_2019 = df_verificacion.filter(col("anio_mes") == "2019-11")
print(f"Filas en 2019-11: {ventas_noviembre_2019.count()}")
ventas_noviembre_2019.show(5, truncate=False)

## Cierre

`transactions_particionado/` (dentro de `artifacts/`) es ahora el dataset base para construir series de tiempo en S4: agrupar por `t_dat` (o por `anio_mes`, para una serie mensual) da la serie de ventas a lo largo del tiempo, ya sin nulos, sin duplicados sin diagnosticar, y particionado para que leer un rango de fechas no implique escanear el dataset completo.

`spark.stop()` si vas a cerrar el notebook aca — libera los recursos del driver antes de pasar a otra cosa.